# BankingClassification Training (Colab + TPU-aware Embeddings)

This notebook runs training in Colab and computes embeddings locally inside Colab.
If TPU is available and `torch_xla` is installed, embeddings run on TPU; otherwise GPU/CPU is used.

In [ ]:
# Install dependencies
!pip -q install --upgrade pip
!pip -q install "torch>=2.1.0" "transformers>=4.40.0" "datasets>=3.0.0" "numpy>=1.24.0" "sentencepiece>=0.2.0"

# Optional: enable TPU support (uncomment when using TPU runtime)
# !pip -q install torch_xla[tpu] -f https://storage.googleapis.com/libtpu-releases/index.html

# Clone project (or update if already present)
import os
if not os.path.exists('/content/BankingClassification'):
    !git clone https://github.com/savg92/BankingClassification.git /content/BankingClassification
else:
    !git -C /content/BankingClassification pull --ff-only


In [ ]:
# Project setup
import os
import sys

os.chdir('/content/BankingClassification')
if '/content/BankingClassification' not in sys.path:
    sys.path.insert(0, '/content/BankingClassification')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ.setdefault('TRAIN_TARGET', 'both')
os.environ.setdefault('TRAIN_CONFIG_LIMIT', '9')
os.environ.setdefault('TRAIN_SLICE', '0')


In [ ]:
# Local embedding model (TPU-aware)
from __future__ import annotations

from typing import Iterable

import torch
from transformers import AutoModel, AutoTokenizer

try:
    import torch_xla.core.xla_model as xm
    XLA_AVAILABLE = True
except Exception:
    xm = None
    XLA_AVAILABLE = False

if XLA_AVAILABLE:
    DEVICE = xm.xla_device()
elif torch.cuda.is_available():
    DEVICE = torch.device('cuda')
else:
    DEVICE = torch.device('cpu')

MODEL_NAME = os.getenv('LOCAL_EMBEDDING_MODEL', 'intfloat/e5-small-v2')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

print(f'Embedding model: {MODEL_NAME}')
print(f'Device: {DEVICE}')

def _mean_pool(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

def embed_texts_local(texts: list[str], batch_size: int = 32) -> list[list[float]]:
    vectors: list[list[float]] = []
    with torch.no_grad():
        for idx in range(0, len(texts), batch_size):
            batch = texts[idx: idx + batch_size]
            # E5 models perform better with query/passsage prefixes
            batch = [f'query: {text}' for text in batch]
            enc = tokenizer(batch, padding=True, truncation=True, max_length=256, return_tensors='pt')
            enc = {k: v.to(DEVICE) for k, v in enc.items()}
            out = model(**enc)
            pooled = _mean_pool(out.last_hidden_state, enc['attention_mask'])
            pooled = torch.nn.functional.normalize(pooled, p=2, dim=1)
            if XLA_AVAILABLE:
                pooled = pooled.cpu()
                xm.mark_step()
            else:
                pooled = pooled.cpu()
            vectors.extend(pooled.tolist())
    return vectors


In [ ]:
# Patch training pipeline to use local embeddings instead of LiteLLM endpoint
from training.banking_classification_training import pipeline as training_pipeline

training_pipeline._embed_texts_with_litellm = embed_texts_local

result = training_pipeline.run_training_pipeline(target=os.getenv('TRAIN_TARGET', 'both'))
print(result)


In [ ]:
# Optional: run individual targets
# result_intent = training_pipeline.run_training_pipeline(target='intent')
# result_sentiment = training_pipeline.run_training_pipeline(target='sentiment')
# print(result_intent)
# print(result_sentiment)
